# Notebook 10: PneumoXNet Grad-CAM Explainability

## Objective

Generate Grad-CAM visualizations for the proposed PneumoXNet model to
show which regions of a chest X-ray the model relies on when making
its BACTERIA / NORMAL / VIRUS prediction.

## Workflow

1. Load the trained PneumoXNet model
2. Implement Grad-CAM (hooked on the Residual Enhancement output --
   PneumoXNet's final spatial feature map before pooling)
3. Visualize a single example
4. Generate representative (highest-confidence, correctly classified)
   Grad-CAM images for each class -- for the paper figure
5. Generate Grad-CAM for misclassified images -- for error analysis
   (this is where the VIRUS/BACTERIA confusion can be visually inspected)
6. Bonus: Baseline (EfficientNet-B2) vs Proposed (PneumoXNet) Grad-CAM
   side-by-side comparison

In [ ]:
# ============================================================
# Cell 1: Import Required Libraries
# ============================================================

from pathlib import Path
import os
import copy

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image

import cv2

import torch
import torch.nn as nn

from torchvision import datasets, transforms

from torchvision.models import (
    efficientnet_b2,
    EfficientNet_B2_Weights
)

sns.set_theme(style="whitegrid")

print("=" * 70)
print("Cell 1 : Libraries Imported Successfully")
print("=" * 70)

## Project Configuration

In [ ]:
# ============================================================
# Cell 2: Project Configuration
# ============================================================

PROJECT_DIR = Path(
    "/mnt/g/Research paper/Research paper/Pneumonia-MultiModel-XAI"
)

DATASET_DIR = PROJECT_DIR / "dataset" / "processed_dataset"

TEST_DIR = DATASET_DIR / "test"

MODEL_PATH = PROJECT_DIR / "models" / "pneumoxnet_best.pth"

# Used only for the optional baseline-vs-proposed comparison figure
BASELINE_MODEL_PATH = PROJECT_DIR / "models" / "efficientnet_b2_best.pth"

RESULTS_DIR = PROJECT_DIR / "results"

GRADCAM_DIR = RESULTS_DIR / "gradcam_pneumoxnet"

GRADCAM_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 70)
print("Cell 2 : Project Configuration")
print("-" * 70)
print("Project Directory  :", PROJECT_DIR)
print("Dataset Directory  :", DATASET_DIR)
print("Model Path         :", MODEL_PATH)
print("Baseline Model Path:", BASELINE_MODEL_PATH)
print("Results Directory  :", GRADCAM_DIR)
print("Device             :", DEVICE)
print("=" * 70)

## Dataset Configuration

In [ ]:
# ============================================================
# Cell 3: Dataset Configuration
# ============================================================

# Must match Notebook 09 (IMAGE_SIZE = 260, EfficientNet-B2 native resolution)
IMAGE_SIZE = 260

NUM_CLASSES = 3

CLASS_NAMES = [
    "BACTERIA",
    "NORMAL",
    "VIRUS"
]

transform = transforms.Compose([

    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )

])

test_dataset = datasets.ImageFolder(
    TEST_DIR,
    transform=transform
)

print("=" * 70)
print("Cell 3 : Dataset Configuration")
print("-" * 70)
print(f"Test Images   : {len(test_dataset)}")
print(f"Classes       : {test_dataset.classes}")
print("=" * 70)

## PneumoXNet Architecture

Rebuilds the exact architecture from Notebook 09 (CBAM, lightweight
Multi-Scale Feature Fusion, Adaptive Feature Fusion, Residual
Enhancement) so the trained weights can be loaded.

In [ ]:
# ============================================================
# Cell 4: PneumoXNet Architecture (must match Notebook 09 exactly)
# ============================================================

# --------------------------------------------------
# Channel Attention
# --------------------------------------------------

class ChannelAttention(nn.Module):

    def __init__(self, in_channels, reduction_ratio=16):

        super(ChannelAttention, self).__init__()

        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)

        self.shared_mlp = nn.Sequential(
            nn.Conv2d(in_channels, in_channels // reduction_ratio, kernel_size=1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels // reduction_ratio, in_channels, kernel_size=1, bias=False)
        )

        self.sigmoid = nn.Sigmoid()

    def forward(self, x):

        avg_out = self.shared_mlp(self.avg_pool(x))
        max_out = self.shared_mlp(self.max_pool(x))

        attention = self.sigmoid(avg_out + max_out)

        return x * attention


# --------------------------------------------------
# Spatial Attention
# --------------------------------------------------

class SpatialAttention(nn.Module):

    def __init__(self, kernel_size=7):

        super(SpatialAttention, self).__init__()

        self.conv = nn.Conv2d(2, 1, kernel_size=kernel_size, padding=kernel_size // 2, bias=False)

        self.sigmoid = nn.Sigmoid()

    def forward(self, x):

        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)

        combined = torch.cat([avg_out, max_out], dim=1)

        attention = self.sigmoid(self.conv(combined))

        return x * attention


# --------------------------------------------------
# CBAM Module
# --------------------------------------------------

class CBAM(nn.Module):

    def __init__(self, in_channels, reduction_ratio=16):

        super(CBAM, self).__init__()

        self.channel_attention = ChannelAttention(in_channels, reduction_ratio)
        self.spatial_attention = SpatialAttention()

    def forward(self, x):

        x = self.channel_attention(x)
        x = self.spatial_attention(x)

        return x


# --------------------------------------------------
# Multi-Scale Feature Fusion (Lightweight, Bottleneck + Dilated Conv)
# --------------------------------------------------

class MultiScaleFeatureFusion(nn.Module):

    def __init__(self, in_channels, reduction=4):

        super(MultiScaleFeatureFusion, self).__init__()

        mid_channels = in_channels // reduction

        self.reduce = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True)
        )

        self.branch_1x1 = nn.Sequential(
            nn.Conv2d(mid_channels, mid_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True)
        )

        self.branch_3x3 = nn.Sequential(
            nn.Conv2d(mid_channels, mid_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True)
        )

        self.branch_5x5 = nn.Sequential(
            nn.Conv2d(mid_channels, mid_channels, kernel_size=3, padding=2, dilation=2, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True)
        )

        self.fusion = nn.Sequential(
            nn.Conv2d(mid_channels * 3, in_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):

        x = self.reduce(x)

        feature_1 = self.branch_1x1(x)
        feature_2 = self.branch_3x3(x)
        feature_3 = self.branch_5x5(x)

        fused = torch.cat([feature_1, feature_2, feature_3], dim=1)

        output = self.fusion(fused)

        return output


# --------------------------------------------------
# Adaptive Feature Fusion
# --------------------------------------------------

class AdaptiveFeatureFusion(nn.Module):

    def __init__(self, channels):

        super().__init__()

        self.weight_generator = nn.Sequential(
            nn.Conv2d(channels * 2, channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, kernel_size=1, bias=False),
            nn.Sigmoid()
        )

    def forward(self, cbam_feature, multiscale_feature):

        fused = torch.cat([cbam_feature, multiscale_feature], dim=1)

        weights = self.weight_generator(fused)

        output = weights * cbam_feature + (1.0 - weights) * multiscale_feature

        return output


# --------------------------------------------------
# Residual Feature Enhancement
# --------------------------------------------------

class ResidualEnhancement(nn.Module):

    def __init__(self, channels):

        super().__init__()

        self.refine = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):

        identity = x

        out = self.refine(x)

        out = out + identity

        return out


# --------------------------------------------------
# PneumoXNet
# --------------------------------------------------

class PneumoXNet(nn.Module):

    def __init__(self, num_classes=3):

        super(PneumoXNet, self).__init__()

        weights = EfficientNet_B2_Weights.DEFAULT
        backbone = efficientnet_b2(weights=weights)

        self.backbone = backbone.features
        self.feature_channels = 1408

        self.cbam = CBAM(in_channels=self.feature_channels)
        self.multiscale = MultiScaleFeatureFusion(in_channels=self.feature_channels)
        self.aff = AdaptiveFeatureFusion(channels=self.feature_channels)
        self.residual = ResidualEnhancement(channels=self.feature_channels)

        self.pool = nn.AdaptiveAvgPool2d(1)

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(self.feature_channels, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):

        features = self.backbone(x)

        cbam_features = self.cbam(features)
        multiscale_features = self.multiscale(features)

        fused_features = self.aff(cbam_features, multiscale_features)

        enhanced_features = self.residual(fused_features)

        pooled_features = self.pool(enhanced_features)

        output = self.classifier(pooled_features)

        return output


print("=" * 70)
print("Cell 4 : PneumoXNet Architecture Rebuilt Successfully")
print("=" * 70)

## Load Trained PneumoXNet Model

In [ ]:
# ============================================================
# Cell 5: Load Trained PneumoXNet Model
# ============================================================

model = PneumoXNet(num_classes=NUM_CLASSES)

state_dict = torch.load(
    MODEL_PATH,
    map_location=DEVICE
)

model.load_state_dict(state_dict)

model = model.to(DEVICE)
model.eval()

print("=" * 70)
print("Cell 5 : PneumoXNet Loaded Successfully")
print("-" * 70)
print("Model Path :", MODEL_PATH)
print("=" * 70)

## Grad-CAM Implementation

In [ ]:
# ============================================================
# Cell 6: Custom Grad-CAM
# ============================================================

class GradCAM:

    def __init__(self, model, target_layer):

        self.model = model
        self.target_layer = target_layer

        self.gradients = None
        self.activations = None

        self.forward_hook = target_layer.register_forward_hook(self.save_activation)
        self.backward_hook = target_layer.register_full_backward_hook(self.save_gradient)

    def save_activation(self, module, input, output):

        self.activations = output.detach()

    def save_gradient(self, module, grad_input, grad_output):

        self.gradients = grad_output[0].detach()

    def remove_hooks(self):

        self.forward_hook.remove()
        self.backward_hook.remove()

print("Cell 6 : Grad-CAM class created.")

## Target Layer

For a plain backbone (Notebook 05), the last convolutional block of
EfficientNet-B2 is used. PneumoXNet processes the backbone features
further through CBAM + Multi-Scale Fusion -> Adaptive Feature Fusion
-> Residual Enhancement before pooling, so the **Residual Enhancement
output** (`model.residual`) is used here instead -- it is PneumoXNet's
final spatial feature map, and reflects the combined effect of all
four proposed modules, not just the raw backbone.

In [ ]:
# ============================================================
# Cell 7: Target Layer
# ============================================================

target_layer = model.residual

gradcam = GradCAM(model, target_layer)

print("=" * 70)
print("Cell 7 : Target Layer Selected (model.residual)")
print("=" * 70)
print(target_layer)

## Load One Test Image

In [ ]:
# ============================================================
# Cell 8: Load One Test Image
# ============================================================

# Change index to visualize a different image
IMAGE_INDEX = 0

image_tensor, true_label = test_dataset[IMAGE_INDEX]

input_tensor = image_tensor.unsqueeze(0).to(DEVICE)

print("=" * 70)
print("Cell 8 : Image Loaded Successfully")
print("-" * 70)
print(f"True Label : {CLASS_NAMES[true_label]}")
print("=" * 70)

## Generate Prediction

In [ ]:
# ============================================================
# Cell 9: Prediction
# ============================================================

with torch.no_grad():

    outputs = model(input_tensor)

    probabilities = torch.softmax(outputs, dim=1)

    confidence, prediction = torch.max(probabilities, 1)

predicted_label = prediction.item()
confidence = confidence.item()

print("=" * 70)
print("Cell 9 : Prediction")
print("-" * 70)
print(f"True Label      : {CLASS_NAMES[true_label]}")
print(f"Predicted Label : {CLASS_NAMES[predicted_label]}")
print(f"Confidence      : {confidence:.4f}")
print("=" * 70)

## Generate Grad-CAM Heatmap

In [ ]:
# ============================================================
# Cell 10: Generate Grad-CAM
# ============================================================

model.zero_grad()

outputs = model(input_tensor)

target = outputs[:, predicted_label]

target.backward()

gradients = gradcam.gradients[0]
activations = gradcam.activations[0]

weights = gradients.mean(dim=(1, 2))

cam = torch.zeros(activations.shape[1:], dtype=torch.float32).to(DEVICE)

for i, w in enumerate(weights):
    cam += w * activations[i]

cam = torch.relu(cam)

cam -= cam.min()

cam /= (cam.max() + 1e-8)

cam = cam.cpu().numpy()

print("Cell 10 : Grad-CAM Heatmap Generated")

In [ ]:
# ============================================================
# Cell 11: Display Grad-CAM
# ============================================================

image = image_tensor.permute(1, 2, 0).numpy()

mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])

image = std * image + mean
image = np.clip(image, 0, 1)

heatmap = cv2.resize(cam, (image.shape[1], image.shape[0]))

heatmap = np.uint8(255 * heatmap)

heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)

heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)

overlay = heatmap.astype(np.float32) / 255 * 0.4 + image * 0.6
overlay = np.clip(overlay, 0, 1)

plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.imshow(image)
plt.title("Original")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(heatmap)
plt.title("Grad-CAM")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(overlay)
plt.title(f"True: {CLASS_NAMES[true_label]}\nPred: {CLASS_NAMES[predicted_label]}")
plt.axis("off")

plt.tight_layout()

plt.savefig(
    GRADCAM_DIR / "pneumoxnet_gradcam_example.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Cell 11 : Grad-CAM image saved successfully.")

## Find Representative Test Images

Highest-confidence, correctly classified image per class -- used for
the paper figure.

In [ ]:
# ============================================================
# Cell 12: Find Best Representative Images
# ============================================================

best_examples = {}

model.eval()

with torch.no_grad():

    for idx in range(len(test_dataset)):

        image, label = test_dataset[idx]

        input_t = image.unsqueeze(0).to(DEVICE)

        output = model(input_t)

        prob = torch.softmax(output, dim=1)

        confidence, prediction = torch.max(prob, 1)

        prediction = prediction.item()
        confidence = confidence.item()

        if prediction != label:
            continue

        if label not in best_examples:

            best_examples[label] = (idx, confidence)

        else:

            if confidence > best_examples[label][1]:

                best_examples[label] = (idx, confidence)

print("=" * 70)
print("Cell 12 : Best Representative Images")
print("-" * 70)

for label in sorted(best_examples):

    idx, conf = best_examples[label]

    print(f"{CLASS_NAMES[label]:10s}  Image #{idx:4d}  Confidence={conf:.4f}")

print("=" * 70)

## Grad-CAM Utility Functions

In [ ]:
# ============================================================
# Cell 13: Grad-CAM Utility Functions
# ============================================================

def generate_gradcam(image_tensor, class_index=None):

    model.zero_grad()

    input_t = image_tensor.unsqueeze(0).to(DEVICE)

    outputs = model(input_t)

    if class_index is None:
        class_index = outputs.argmax(dim=1).item()

    outputs[:, class_index].backward()

    gradients = gradcam.gradients[0]
    activations = gradcam.activations[0]

    weights = gradients.mean(dim=(1, 2))

    cam = torch.zeros(activations.shape[1:], device=DEVICE)

    for i, w in enumerate(weights):
        cam += w * activations[i]

    cam = torch.relu(cam)

    cam -= cam.min()

    cam /= (cam.max() + 1e-8)

    return cam.cpu().numpy(), class_index


def tensor_to_image(image_tensor):

    image = image_tensor.permute(1, 2, 0).numpy()

    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])

    image = image * std + mean

    image = np.clip(image, 0, 1)

    return image


def overlay_heatmap(image, cam):

    heatmap = cv2.resize(cam, (image.shape[1], image.shape[0]))

    heatmap = np.uint8(255 * heatmap)

    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)

    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)

    overlay = image * 0.6 + heatmap.astype(np.float32) / 255 * 0.4

    overlay = np.clip(overlay, 0, 1)

    return heatmap, overlay


print("Cell 13 : Grad-CAM utility functions created.")

## Generate & Save Representative Grad-CAM Images (Paper Figure)

In [ ]:
# ============================================================
# Cell 14: Generate & Save Representative Grad-CAM Images
# ============================================================

fig, axes = plt.subplots(NUM_CLASSES, 3, figsize=(15, 15))

for row, label in enumerate(sorted(best_examples.keys())):

    image_index, confidence = best_examples[label]

    image_tensor, true_label = test_dataset[image_index]

    cam, predicted_class = generate_gradcam(image_tensor)

    original = tensor_to_image(image_tensor)

    heatmap, overlay = overlay_heatmap(original, cam)

    axes[row, 0].imshow(original)
    axes[row, 0].set_title(f"{CLASS_NAMES[true_label]}\nOriginal")
    axes[row, 0].axis("off")

    axes[row, 1].imshow(heatmap)
    axes[row, 1].set_title("Grad-CAM")
    axes[row, 1].axis("off")

    axes[row, 2].imshow(overlay)
    axes[row, 2].set_title(f"Pred: {CLASS_NAMES[predicted_class]}\nConf: {confidence:.3f}")
    axes[row, 2].axis("off")

    plt.imsave(
        GRADCAM_DIR / f"pneumoxnet_{CLASS_NAMES[true_label].lower()}_correct.png",
        overlay
    )

plt.tight_layout()

plt.savefig(
    GRADCAM_DIR / "pneumoxnet_gradcam_paper_figure.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("=" * 70)
print("Cell 14 : Representative Grad-CAM Images Generated")
print("Saved to:", GRADCAM_DIR)
print("=" * 70)

## Misclassified Grad-CAM Analysis

This is especially useful for PneumoXNet, since VIRUS-vs-BACTERIA
confusion was the main weakness found during evaluation (Notebook 09)
-- inspecting where the model looks on these errors can reveal
whether it is focusing on relevant lung regions at all.

In [ ]:
# ============================================================
# Cell 15: Generate Grad-CAM for Misclassified Images
# ============================================================

misclassified_examples = []

model.eval()

with torch.no_grad():

    for idx in range(len(test_dataset)):

        image_tensor, true_label = test_dataset[idx]

        input_t = image_tensor.unsqueeze(0).to(DEVICE)

        outputs = model(input_t)

        probabilities = torch.softmax(outputs, dim=1)

        confidence, prediction = torch.max(probabilities, 1)

        prediction = prediction.item()
        confidence = confidence.item()

        if prediction != true_label:

            misclassified_examples.append((idx, true_label, prediction, confidence))

print(f"Total Misclassified Images: {len(misclassified_examples)}")

# Display first 6 VIRUS<->BACTERIA misclassifications specifically,
# falling back to the first 6 misclassifications overall if fewer exist
priority = [
    e for e in misclassified_examples
    if {e[1], e[2]} == {0, 2}   # BACTERIA (0) <-> VIRUS (2)
]

display_examples = (priority if len(priority) >= 6 else misclassified_examples)

num_examples = min(6, len(display_examples))

fig, axes = plt.subplots(num_examples, 3, figsize=(15, 5 * num_examples))

if num_examples == 1:
    axes = np.expand_dims(axes, axis=0)

for row in range(num_examples):

    idx, true_label, pred_label, confidence = display_examples[row]

    image_tensor, _ = test_dataset[idx]

    cam, _ = generate_gradcam(image_tensor, pred_label)

    original = tensor_to_image(image_tensor)

    heatmap, overlay = overlay_heatmap(original, cam)

    axes[row, 0].imshow(original)
    axes[row, 0].set_title(f"Original\nTrue: {CLASS_NAMES[true_label]}")
    axes[row, 0].axis("off")

    axes[row, 1].imshow(heatmap)
    axes[row, 1].set_title("Grad-CAM")
    axes[row, 1].axis("off")

    axes[row, 2].imshow(overlay)
    axes[row, 2].set_title(f"Pred: {CLASS_NAMES[pred_label]}\nConf: {confidence:.3f}")
    axes[row, 2].axis("off")

plt.tight_layout()

plt.savefig(
    GRADCAM_DIR / "pneumoxnet_misclassified_gradcam.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Cell 15 : Misclassified Grad-CAM figure saved successfully.")

## Bonus: Baseline (EfficientNet-B2) vs Proposed (PneumoXNet) Comparison

Loads the plain EfficientNet-B2 baseline model (Notebook 03) alongside
PneumoXNet and compares Grad-CAM on the same image. This is a useful
figure for the paper's discussion section -- it can visually support
(or challenge) the claim that the added attention/fusion modules focus
on more clinically relevant regions than the baseline backbone alone.

In [ ]:
# ============================================================
# Cell 16: Load Baseline EfficientNet-B2 for Comparison
# ============================================================

baseline_model = efficientnet_b2(weights=EfficientNet_B2_Weights.DEFAULT)

in_features = baseline_model.classifier[1].in_features

baseline_model.classifier[1] = nn.Sequential(
    nn.Dropout(p=0.4),
    nn.Linear(in_features, NUM_CLASSES)
)

baseline_checkpoint = torch.load(
    BASELINE_MODEL_PATH,
    map_location=DEVICE
)

if isinstance(baseline_checkpoint, dict) and "model_state_dict" in baseline_checkpoint:
    baseline_model.load_state_dict(baseline_checkpoint["model_state_dict"])
else:
    baseline_model.load_state_dict(baseline_checkpoint)

baseline_model = baseline_model.to(DEVICE)
baseline_model.eval()

baseline_target_layer = baseline_model.features[-1]

baseline_gradcam = GradCAM(baseline_model, baseline_target_layer)

print("=" * 70)
print("Cell 16 : Baseline EfficientNet-B2 Loaded for Comparison")
print("=" * 70)

In [ ]:
# ============================================================
# Cell 17: Baseline vs Proposed Grad-CAM Comparison
# ============================================================

def generate_gradcam_for(model_obj, gradcam_obj, image_tensor, class_index=None):

    model_obj.zero_grad()

    input_t = image_tensor.unsqueeze(0).to(DEVICE)

    outputs = model_obj(input_t)

    if class_index is None:
        class_index = outputs.argmax(dim=1).item()

    outputs[:, class_index].backward()

    gradients = gradcam_obj.gradients[0]
    activations = gradcam_obj.activations[0]

    weights = gradients.mean(dim=(1, 2))

    cam = torch.zeros(activations.shape[1:], device=DEVICE)

    for i, w in enumerate(weights):
        cam += w * activations[i]

    cam = torch.relu(cam)

    cam -= cam.min()

    cam /= (cam.max() + 1e-8)

    return cam.cpu().numpy(), class_index


COMPARISON_IMAGE_INDEX = 0  # change to compare a different test image

image_tensor, true_label = test_dataset[COMPARISON_IMAGE_INDEX]

original = tensor_to_image(image_tensor)

base_cam, base_pred = generate_gradcam_for(baseline_model, baseline_gradcam, image_tensor)
prop_cam, prop_pred = generate_gradcam_for(model, gradcam, image_tensor)

_, base_overlay = overlay_heatmap(original, base_cam)
_, prop_overlay = overlay_heatmap(original, prop_cam)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(original)
axes[0].set_title(f"Original\nTrue: {CLASS_NAMES[true_label]}")
axes[0].axis("off")

axes[1].imshow(base_overlay)
axes[1].set_title(f"Baseline (EfficientNet-B2)\nPred: {CLASS_NAMES[base_pred]}")
axes[1].axis("off")

axes[2].imshow(prop_overlay)
axes[2].set_title(f"Proposed (PneumoXNet)\nPred: {CLASS_NAMES[prop_pred]}")
axes[2].axis("off")

plt.tight_layout()

plt.savefig(
    GRADCAM_DIR / "baseline_vs_proposed_gradcam.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Cell 17 : Baseline vs Proposed comparison figure saved successfully.")

## Conclusion

In [ ]:
# ============================================================
# Cell 18: Notebook Completed
# ============================================================

print("=" * 70)
print("Notebook 10 Completed Successfully")
print("=" * 70)

print("Generated Outputs:")
print("  pneumoxnet_gradcam_example.png")
print("  pneumoxnet_gradcam_paper_figure.png")
print("  pneumoxnet_bacteria_correct.png")
print("  pneumoxnet_normal_correct.png")
print("  pneumoxnet_virus_correct.png")
print("  pneumoxnet_misclassified_gradcam.png")
print("  baseline_vs_proposed_gradcam.png")

print("\nLocation:")
print(GRADCAM_DIR)

print("=" * 70)